<hr style="border:2px solid black"> </hr>
<h2 style="color:Crimson;">L03X.1 Bonus Notebook</h2>

### **Copies**

**Reference:** https://docs.python.org/3.13/library/copy.html

In this notebook, we show the effect of different techniques for *copying objects in Python*.

**Inspect** these operations, and explore the characteristics of the individual components.

In [ ]:
d = dict(((1, 1), (2, [2])))
d

In [ ]:
id(d)                  # the memory ID for this specific object (will vary)

In [ ]:
import copy

In [ ]:
db = d                 # copy by assignment
dc = d.copy()          # shallow copy, new memory IDs for top-level contents
dd = copy.deepcopy(d)  # deep copy, new memory IDs for all contents, recursively

In [ ]:
d[1] = 'one'
d[2][0] = 'two'

In [ ]:
d

In [ ]:
db

In [ ]:
dc

In [ ]:
dd

In [ ]:
d[2] = 'three'
d

In [ ]:
db

In [ ]:
dc

In [ ]:
dd

In [ ]:
id(d), id(db), id(dc), id(dd)

In [ ]:
# For deeper ID inspection

for name, dy in zip(['d', 'db', 'dc', 'dd'], [d, db, dc, dd]):
    print(f'{name :<4}- {id(dy)}')
    for k in dy.keys():
        print(f'  {k}:{str(dy[k]) :<10} {id(dy[k])}')

-----
**Notice anything confusing?**

Try this notebook again with `d = {1: [1, 2], 2: [3, 4]}`

Can you work out why the results are different, particularly for the ID of the value at key `1` for `dc` and `dd`?

-----

### A Deeper Look at `deepcopy`

`deepcopy` is implemented in pure Python from the `copy` library.  When invoked, it crawls ***every attribute of every nested object***, consulting a memo dictionary to avoid infinite recursion. 

On modest payloads this is tolerable; on realistic data structures (100,000-plus nodes, `pandas` objects, `numpy` arrays, large strings) it is frequently the slowest operation in a profile. Worse, the time cost is accompanied by a **second full memory** footprint.  In memory-bound data science and HPC contexts, that doubling is rarely acceptable.  

Sometimes we can get in the **habit** of saying things like "but my data structure is only a few kilobytes".  This is fair, but once the habit is baked in, it tends to persist well into the future as the codebase scales.  Often, maintainers would rather **prevent the habit** than police each use.

Deep copying looks like a guarantee of object separation/independence, but the guarantee is **partial and brittle**.

- File handles, sockets, CUDA contexts and similar items cannot be duplicated meaningfully.  After a deep copy, you may end up with two Python objects that both think they own the same operating system-level resource.

- Many libraries use **identity** as part of their design contract (e.g., caching, logging handlers and TensorFlow graphs).  Deep copies in this instance creates clones that can break assumptions and even create subtle data races.

- Unless a class implements its own `__deepcopy__`, the default heuristic is a shallow copy of `__dict__`.  Any invariants enforced in `__init__` are bypassed, so caches, locks, or validators may never run.

These problems are generally **silent**; the code runs and only misbehaves later.

-----

Reaching for `deepcopy` often signals that the coded data model is **mutable** when it could be differently designed.
- If data is immutable, copies are cheap (just new references), and the mental model is far clearer.
- A "clone and tweak" pattern could be cleaner, as `new_obj = ExistingType(old.a, b=42)` or, with `@dataclass`, `replace(old, b=42)`.
- Sometimes we might actually want two structures to share an internal list; duplicating it hides the coupling and can mask bugs rather than fix them.

-----

Debuggability and maintainability can also be a problem.
- A deep copy inserts an **opaque, non-local** transformation: after the call you hold objects whose identities, class invariants, and even lineage are no longer evident in the stack trace.
- The next maintainer must mentally simulate what might have been copied or read the source of every class involved.
- In contrast, an **explicit** constructor call, comprehension, or `pandas.DataFrame.copy(deep=True)` each states its intent and scope plainly.

----- 

When `deepcopy` is generally ok:
- Throw-away scripts, REPL experiments, *unit tests where correctness matters more than speed*.
- Small nested `dict`/`list` structures representing pure data and *nothing with external side-effects.*
- Legacy code you must patch quickly and cannot refactor all of.


-----

Safer, more idiomatic alternatives:

Use-case|Typical substitute
--:|:---
Copy a dict or list one level deep |	`.copy() / lst[:]`
Clone a dataclass changing one field |	`dataclasses.replace(obj, field=new)`
Duplicate a NumPy array	| `arr.copy()` (respects memory order)
Clone a pandas `DataFrame` |	`df.copy(deep=True)`
Serialise / deserialise for a true deep copy of simple types |	`json.loads(json.dumps(obj))` (if JSON-serialisable)
Immutable data with tweaks |	`attrs`' or pydantic’s `.copy(update={...})`, or in 3.13: `copy.replace`

These are **faster, explicit** about what will and will not be shared, and generally free of the semantic issues above.


-----

`deepcopy` being declared **wrong** is probably overstated.  Like many exaggerations, though, it functions as a heuristic.  Nine times out of ten,  code does become clearer, faster and safer once the `deepcopy` is removed and the underlying data model flaw is fixed.  Reserving `deepcopy` for that tenth case may save you from the other nine.

